In [1]:
from langchain_groq import ChatGroq

In [98]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='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', 
    model_name="llama-3.3-70b-versatile"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [100]:
!pip install langchain_community

In [124]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://about.puma.com/en/jobs/area-retail-manager-hyderabad-r42275")
page_data = loader.load().pop().page_content
print(page_data)




















Area Retail Manager - Hyderabad














































				Skip to main content
			















			Menu
			










						Back
					






This is PUMA 


Overview

Our Sports 


Overview

						Football and Other Teamsports
											


						Track and Field
											


						Motorsport
											


						Golf
											


						Basketball
											





						Our Management
											


						History
											


						Archive Stories
											





Newsroom 


Overview

						News
											


						Images and Footage
											


						Media Contacts
											


						News Calendar
											





Investor Relations 


Overview

Share 



						Share Price
											


						Analyst Coverage
											


						Share Buyback 2024 - 2025
											


						Shareholder Structure
											


						Voting Rights Notifications
											


						Directors' Dealings
											





						Investor & Ad-Hoc News
									

In [126]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [128]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'role': 'Area Retail Manager - Hyderabad',
  'experience': '5+ years',
  'skills': ['Excellent communication skills', 'Strong analytical skills'],
  'description': 'Drive sales growth, profitability, and operational excellence across the assigned region comprising Full Price (FP) and Factory Outlet Centre (FOC) stores.'},
 {'role': 'Area Retail Manager-North',
  'experience': None,
  'skills': None,
  'description': 'Retail Management / Operations'},
 {'role': 'Cluster Visual Merchandising - Upper North (Third Party)',
  'experience': None,
  'skills': None,
  'description': 'Visual Merchandising'},
 {'role': 'Area Visual Merchandising',
  'experience': None,
  'skills': None,
  'description': 'Visual Merchandising'}]

In [130]:
type(json_res)

list

In [132]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [134]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [136]:
links = collection.query(query_texts=["Experience in Python","Expertise in React Native"], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/react-native-portfolio'},
  {'links': 'https://example.com/react-portfolio'}]]

In [140]:
job = json_res
job['skills']

TypeError: list indices must be integers or slices, not str

In [142]:
[job["skills"] for job in json_res]

[['Excellent communication skills', 'Strong analytical skills'],
 None,
 None,
 None]

In [144]:
job

[{'role': 'Area Retail Manager - Hyderabad',
  'experience': '5+ years',
  'skills': ['Excellent communication skills', 'Strong analytical skills'],
  'description': 'Drive sales growth, profitability, and operational excellence across the assigned region comprising Full Price (FP) and Factory Outlet Centre (FOC) stores.'},
 {'role': 'Area Retail Manager-North',
  'experience': None,
  'skills': None,
  'description': 'Retail Management / Operations'},
 {'role': 'Cluster Visual Merchandising - Upper North (Third Party)',
  'experience': None,
  'skills': None,
  'description': 'Visual Merchandising'},
 {'role': 'Area Visual Merchandising',
  'experience': None,
  'skills': None,
  'description': 'Visual Merchandising'}]

In [146]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Shreys, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Enhance Your Retail Operations with AtliQ's Automated Solutions

Dear Hiring Manager,

I came across the job descriptions for Area Retail Manager, Cluster Visual Merchandising, and Area Visual Merchandising roles at your esteemed organization. As a Business Development Executive at AtliQ, I believe our AI and software consulting services can significantly enhance your retail operations, driving sales growth, profitability, and operational excellence.

At AtliQ, we specialize in developing tailored solutions that foster scalability, process optimization, cost reduction, and heightened overall efficiency. Our expertise in automation can help streamline your retail management, visual merchandising, and operational processes. With our solutions, you can expect improved communication, enhanced analytical capabilities, and data-driven decision-making.

Our portfolio showcases our capabilities in machine learning, Python development, React, and React Native. Some notable examples inc